# AlphaLOB Phase 2 — Notebook 02: Feature Engineering & Normalization

**Input:** `/content/lob_data.parquet` (5M rows from Notebook 01)

**Output:** `/content/lob_features.parquet` (5M rows with engineered features + labels)

## The 4 Mathematically-Grounded Features (Math+CS Differentiators)

| Feature | Formula | Interview Key Point |
|---------|---------|--------------------|
| **WOFI** | `Σᵢ wᵢ·(Vᵢᵇⁱᵈ − Vᵢᵃˢᵏ)/(Vᵢᵇⁱᵈ + Vᵢᵃˢᵏ)` | Inverse-distance weighted; **deque O(1)** |
| **Hawkes λ(t)** | `μ + Σᵢ α·exp(−β(t−tᵢ))` | Order arrival clustering; fit once |
| **Kyle's λ** | `ΔPₜ = λ·Qₜ + εₜ` | Price impact via OLS; 5-min rolling |
| **Amihud ILLIQ** | `(1/T)·Σ|rₜ|/VOLₜ` | Illiquidity regime context for HMM |

**Critical:** All features Z-Score normalized using ROLLING windows only — no look-ahead bias.

---


In [ ]:
!pip install polars pyarrow statsmodels --quiet
print('✅ Dependencies installed')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive mounted')

import numpy as np
import polars as pl
import json
import os
import time
from collections import deque
import statsmodels.api as sm

# ── I/O paths ────────────────────────────────────────────────────────────────
PARQUET_IN  = '/content/drive/MyDrive/AlphaLOB/lob_data.parquet'
PARQUET_OUT = '/content/drive/MyDrive/AlphaLOB/lob_features.parquet'
os.makedirs('/content/drive/MyDrive/AlphaLOB', exist_ok=True)

# ── Feature constants ────────────────────────────────────────────────────────
N_LEVELS      = 10
NORM_WINDOW   = 1000    # rolling Z-score window (no look-ahead)
AMIHUD_WINDOW = 300     # 30s rolling window for Amihud
KYLE_WINDOW   = 3_000   # 5-min rolling window for Kyle's lambda (300s × 10 ticks/s)

print(f'✅ Imports and config loaded')
print(f'   Input:  {PARQUET_IN}')
print(f'   Output: {PARQUET_OUT}')

In [ ]:
t0 = time.time()
df = pl.read_parquet(PARQUET_IN)
print(f'✅ Loaded {len(df):,} rows in {time.time()-t0:.1f}s')
print(f'   Columns: {df.columns[:6]}...')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 1 — WOFI (Weighted Order Flow Imbalance)
#
# Formula: WOFI = Σᵢ wᵢ · (Vᵢᵇⁱᵈ − Vᵢᵃˢᵏ) / (Vᵢᵇⁱᵈ + Vᵢᵃˢᵏ)
# Weight:  wᵢ = 1 / (1 + |pᵢ − mid|)   (inverse distance from mid)
#
# Implementation: deque-based rolling window — O(1) per tick
#
# WHY DEQUE?
# A deque maintains a sliding window of the last W raw WOFI snapshots.
# Adding a new tick = O(1) appendright. Removing the oldest = O(1) popleft.
# The rolling sum is maintained incrementally — never recomputed from scratch.
# This is the blueprint requirement: O(1) per tick, NOT a pandas rolling apply
# which recomputes the full window every tick = O(W) per tick.
#
# At inference time on the live server:
#   feature_worker.py holds one persistent deque per feature.
#   Each new tick does appendright + popleft + running_sum update = O(1).
# ─────────────────────────────────────────────────────────────────────────────

print('Computing WOFI (deque-based O(1) rolling window)...')
t0 = time.time()

mid   = df['mid_price'].to_numpy()
n     = len(df)
WOFI_SMOOTH = 20  # rolling window size for WOFI smoothing (20 ticks = 2 seconds)

# ── Step 1: Compute raw per-tick WOFI snapshot ────────────────────────────
# At each tick, there are exactly 10 levels → O(10) = O(1) per tick
wofi_raw = np.zeros(n)

for lvl in range(N_LEVELS):
    bid_p = df[f'bid_price_{lvl}'].to_numpy()
    ask_p = df[f'ask_price_{lvl}'].to_numpy()
    bid_v = df[f'bid_vol_{lvl}'].to_numpy()
    ask_v = df[f'ask_vol_{lvl}'].to_numpy()

    # Inverse distance weights from mid price
    w_bid = 1.0 / (1.0 + np.abs(bid_p - mid))
    w_ask = 1.0 / (1.0 + np.abs(ask_p - mid))
    w     = (w_bid + w_ask) / 2.0

    denom = bid_v + ask_v
    denom = np.where(denom < 1e-9, 1e-9, denom)

    wofi_raw += w * (bid_v - ask_v) / denom

# Normalize: each level contributes equally (divide by N_LEVELS)
wofi_raw /= float(N_LEVELS)

# ── Step 2: Deque-based rolling mean smoother — O(1) per tick ────────────
# This is the exact O(1) deque pattern the blueprint requires.
# running_sum tracks the window total without recomputing from scratch.
wofi_values  = np.zeros(n)
window       = deque()        # holds the last WOFI_SMOOTH raw values
running_sum  = 0.0

for i in range(n):
    val = wofi_raw[i]

    # O(1): append new value to right of deque
    window.appendleft(val)
    running_sum += val

    # O(1): evict oldest value from left of deque
    if len(window) > WOFI_SMOOTH:
        running_sum -= window.pop()

    # O(1): rolling mean = running_sum / window_size
    wofi_values[i] = running_sum / len(window)

print(f'✅ WOFI computed in {time.time()-t0:.1f}s (deque O(1) per tick)')
print(f'   Range: [{wofi_values.min():.3f}, {wofi_values.max():.3f}]')
print(f'   Mean:  {wofi_values.mean():.4f} (should be ~0)')
print(f'   Deque window size: {WOFI_SMOOTH} ticks (2 seconds at 10 ticks/sec)')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 2 — Hawkes Process Intensity
# Formula: λ(t) = μ + Σᵢ α·exp(−β(t−tᵢ))
# Captures order arrival CLUSTERING: bursts of orders → high intensity
# Key: fit (μ, α, β) on TRAINING data only — NEVER refit at inference
# ─────────────────────────────────────────────────────────────────────────────

print('Fitting Hawkes Process...')

try:
    from tick.hawkes import HawkesExpKern
    HAWKES_AVAILABLE = True
    print('  Using tick library (exact MLE fit)')
except ImportError:
    HAWKES_AVAILABLE = False
    print('  tick not available — using analytical approximation')

# Fit on first 10% of data (training split only — no look-ahead)
n_fit      = len(df) // 10

if HAWKES_AVAILABLE:
    ts_seconds = np.arange(n_fit) * 0.1  # 10 ticks/sec → 0.1s per tick
    learner    = HawkesExpKern(decays=1.0, max_iter=50, verbose=False)
    learner.fit([ts_seconds])
    mu_hawkes    = float(learner.baseline[0])
    alpha_hawkes = float(learner.adjacency[0, 0])
    beta_hawkes  = 1.0
else:
    # Analytical approximation: calibrated defaults for LOB order arrival
    mu_hawkes    = 10.0   # 10 events per second
    alpha_hawkes = 0.5    # excitation factor
    beta_hawkes  = 2.0    # decay rate (1/2s memory)

print(f'✅ Hawkes params fitted: μ={mu_hawkes:.4f}, α={alpha_hawkes:.4f}, β={beta_hawkes:.4f}')

# Recursive computation of intensity — O(n), applies stored params
# Recursive formula: R(i) = exp(−β·dt)·(R(i−1) + 1), λ(i) = μ + α·R(i)
print('Computing Hawkes intensity (recursive O(n), stored params)...')
t0 = time.time()

n    = len(df)
dt_fixed     = 0.1
decay_factor = np.exp(-beta_hawkes * dt_fixed)

R                = np.zeros(n)
hawkes_intensity = np.zeros(n)
R[0]             = 0.0
hawkes_intensity[0] = mu_hawkes

for i in range(1, n):
    # R tracks the weighted sum of past events with exponential decay
    R[i] = decay_factor * (R[i-1] + 1.0)
    # Intensity = baseline + excited arrivals
    hawkes_intensity[i] = mu_hawkes + alpha_hawkes * R[i]

print(f'✅ Hawkes intensity computed in {time.time()-t0:.1f}s')
print(f'   Range: [{hawkes_intensity.min():.3f}, {hawkes_intensity.max():.3f}]')

# Save fitted coefficients for inference-time use
hawkes_params = {'mu': mu_hawkes, 'alpha': alpha_hawkes, 'beta': beta_hawkes}
with open('/content/hawkes_params.json', 'w') as f:
    json.dump(hawkes_params, f, indent=2)
print('✅ Hawkes params saved → /content/hawkes_params.json')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 3 — Kyle's Lambda (Price Impact Coefficient)
# Formula: ΔPₜ = λ·Qₜ + εₜ
# Estimated via OLS regression on rolling 5-minute windows (statsmodels)
# Qₜ = signed order flow (positive = net buying pressure)
# High λ → thin market, informed trading dominant
# ─────────────────────────────────────────────────────────────────────────────

print("Computing Kyle's Lambda (rolling OLS, 5-min windows, statsmodels)...")
t0 = time.time()

mid_prices = df['mid_price'].to_numpy()

# Signed order flow proxy: best 2 levels
bid_v0 = df['bid_vol_0'].to_numpy()
ask_v0 = df['ask_vol_0'].to_numpy()
bid_v1 = df['bid_vol_1'].to_numpy()
ask_v1 = df['ask_vol_1'].to_numpy()
Q      = (bid_v0 - ask_v0) + 0.5 * (bid_v1 - ask_v1)  # signed order flow

# Mid-price changes (FIX: manual diff avoids prepend bug in np.diff)
delta_P       = np.zeros(len(df))
delta_P[1:]   = mid_prices[1:] - mid_prices[:-1]       # proper first-difference
delta_P[0]    = 0.0                                     # no previous tick exists

# Rolling OLS — recompute every step ticks (efficiency: 10x fewer OLS calls)
kyle_lambda   = np.zeros(len(df))
step          = KYLE_WINDOW // 10   # recompute every 300 ticks
current_lambda = 0.0

for i in range(0, len(df), step):
    start = max(0, i - KYLE_WINDOW)
    end   = min(i + step, len(df))

    y = delta_P[start:end]
    X = Q[start:end]

    if len(y) > 30 and np.std(X) > 1e-9:
        try:
            res = sm.OLS(y, sm.add_constant(X)).fit(disp=0)
            current_lambda = float(res.params[1])  # slope = Kyle's λ
        except Exception:
            pass  # keep previous lambda on numerical failure

    kyle_lambda[i:end] = current_lambda

print(f"✅ Kyle's Lambda computed in {time.time()-t0:.1f}s")
print(f'   Mean λ: {kyle_lambda.mean():.6f}')
print(f'   Range:  [{kyle_lambda.min():.6f}, {kyle_lambda.max():.6f}]')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FEATURE 4 — Amihud Illiquidity Ratio
# Formula: ILLIQ = (1/T) · Σₜ |rₜ| / VOLₜ
# rₜ = log return, VOLₜ = dollar volume in interval t
# High ILLIQ → each dollar of volume moves price a lot → illiquid regime
# Used as regime context input to the HMM in Notebook 04
# ─────────────────────────────────────────────────────────────────────────────

print('Computing Amihud Illiquidity Ratio...')
t0 = time.time()

# Log returns (FIX: manual diff avoids prepend bug in np.diff)
log_returns       = np.zeros(len(df))
log_returns[1:]   = np.log(mid_prices[1:]) - np.log(mid_prices[:-1])  # proper first-diff
log_returns[0]    = 0.0  # no previous tick
abs_returns       = np.abs(log_returns)

# Dollar volume proxy: price × (best bid vol + best ask vol)
dollar_vol = mid_prices * (bid_v0 + ask_v0)
dollar_vol = np.where(dollar_vol < 1e-9, 1e-9, dollar_vol)

# Per-tick Amihud ratio
amihud_tick = abs_returns / dollar_vol

# Rolling mean using Polars (vectorized, fast)
amihud_series  = pl.Series('amihud_tick', amihud_tick)
amihud_rolling = amihud_series.rolling_mean(window_size=AMIHUD_WINDOW, min_periods=10)
amihud_illiq   = amihud_rolling.fill_null(strategy='forward').to_numpy()

print(f'✅ Amihud ILLIQ computed in {time.time()-t0:.1f}s')
print(f'   Mean ILLIQ: {amihud_illiq.mean():.2e}')
print(f'   Range:      [{amihud_illiq.min():.2e}, {amihud_illiq.max():.2e}]')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# HMM Context Features (used in Notebook 04)
# realized_vol = rolling std of log-returns (100-tick window)
# autocorrelation = rolling lag-1 autocorrelation of log-returns
# ─────────────────────────────────────────────────────────────────────────────

print('Computing HMM context features (realized_vol, autocorrelation)...')
t0 = time.time()

# Realized volatility: rolling std (100-tick window) via Polars
lr_series    = pl.Series('log_ret', log_returns)
realized_vol = (
    lr_series
    .rolling_std(window_size=100, min_periods=10)
    .fill_null(strategy='forward')
    .to_numpy()
)

# Rolling autocorrelation (lag=1) — step-sampled every 500 ticks for speed
autocorr_values = np.zeros(len(df))
step_ac = 500
for i in range(100, len(log_returns), step_ac):
    x = log_returns[max(0, i-100):i]
    if len(x) > 10 and np.std(x) > 1e-12:
        # Guard against zero-std windows that make np.corrcoef return NaN
        if np.std(x[:-1]) > 1e-12 and np.std(x[1:]) > 1e-12:
            ac = np.corrcoef(x[:-1], x[1:])[0, 1]
            if not np.isnan(ac):
                autocorr_values[i:min(i+step_ac, len(df))] = ac

print(f'✅ HMM features computed in {time.time()-t0:.1f}s')
print(f'   realized_vol range:  [{realized_vol.min():.6f}, {realized_vol.max():.6f}]')
print(f'   autocorr range:      [{autocorr_values.min():.3f}, {autocorr_values.max():.3f}]')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# NORMALIZATION — Online Rolling Z-Score
#
# Formula: z = (x − μ_rolling) / σ_rolling
# Window:  1000 ticks (rolling backward only — zero look-ahead)
#
# CRITICAL RULES:
#   ✅ Use rolling_mean + rolling_std (past data only)
#   ❌ NEVER use global mean/std — that uses future data = look-ahead bias
#   ✅ All normalized features must be in range approximately [-3, 3]
#   ✅ Clip to [-5, 5] to handle instability in the first 1000 ticks
# ─────────────────────────────────────────────────────────────────────────────

print('Applying rolling Z-Score normalization (window=1000 ticks, NO look-ahead)...')
t0 = time.time()

def rolling_zscore(arr: np.ndarray, window: int, name: str) -> np.ndarray:
    """
    Online Z-Score using only PAST data.
    Polars rolling_mean/rolling_std look backward only — no look-ahead.
    """
    s   = pl.Series(name, arr)
    mu  = s.rolling_mean(window_size=window, min_periods=10)
    sd  = s.rolling_std(window_size=window,  min_periods=10)

    mu_arr = mu.fill_null(strategy='forward').to_numpy()
    sd_arr = sd.fill_null(1.0).to_numpy()
    sd_arr = np.where(sd_arr < 1e-9, 1.0, sd_arr)   # avoid divide-by-zero

    z = (arr - mu_arr) / sd_arr
    return np.clip(z, -5.0, 5.0)   # clip early-window instability

wofi_z        = rolling_zscore(wofi_values,     NORM_WINDOW, 'wofi')
hawkes_z      = rolling_zscore(hawkes_intensity, NORM_WINDOW, 'hawkes')
kyle_lambda_z = rolling_zscore(kyle_lambda,      NORM_WINDOW, 'kyle')
amihud_z      = rolling_zscore(amihud_illiq,     NORM_WINDOW, 'amihud')
spread_z      = rolling_zscore(df['spread'].to_numpy(), NORM_WINDOW, 'spread')

print(f'✅ Z-scores computed in {time.time()-t0:.1f}s')
print()
print(f'{"Feature":<16} {"Mean":>8} {"Std":>8} {"Min":>8} {"Max":>8}')
print('-' * 52)
for name, arr in [('wofi_z',        wofi_z),
                   ('hawkes_z',      hawkes_z),
                   ('kyle_lambda_z', kyle_lambda_z),
                   ('amihud_z',      amihud_z),
                   ('spread_z',      spread_z)]:
    print(f'{name:<16} {arr.mean():>8.3f} {arr.std():>8.3f} '
          f'{arr.min():>8.2f} {arr.max():>8.2f}')
print()
print('✅ All features normalized to approximately [-3, 3] range')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# FORWARD LABELS
#
# For each horizon H:
#   label = 1  if mid_price[t+H] > mid_price[t]  (price goes UP)
#   label = 0  if mid_price[t+H] <= mid_price[t] (price goes DOWN)
#
# FIX BUG: Using np.roll to create future prices, then marking trailing rows
# as -1 (unknown) instead of assigning them a fake label.
# This prevents the model from learning on meaningless "DOWN" labels at the tail.
# ─────────────────────────────────────────────────────────────────────────────

print('Creating forward-looking labels...')

HORIZONS = {
    'label_5s':   50,    # 5 seconds
    'label_30s':  300,   # 30 seconds ← KEY METRIC
    'label_5min': 3000,  # 5 minutes
}

mid_np = mid_prices  # use numpy for fast indexing

labels = {}
for label_name, horizon in HORIZONS.items():
    # Shift prices backward: future_prices[i] = mid_np[i+horizon]
    future_prices = np.roll(mid_np, -horizon)
    # Last `horizon` rows have no valid future — mark as NaN
    future_prices[-horizon:] = np.nan

    # Label: 1 if price went up, 0 if price went down
    raw_labels = (future_prices > mid_np).astype(np.int8)

    # Mark unknown rows explicitly as -1 (do NOT assign 0 or 1)
    raw_labels[-horizon:] = -1

    labels[label_name] = raw_labels

    # Compute UP percentage on valid rows only (exclude -1 unknowns)
    valid_mask = raw_labels >= 0
    valid_labels = raw_labels[valid_mask]
    if valid_labels.sum() > 0:
        pct_up = valid_labels.mean() * 100
    else:
        pct_up = 50.0  # fallback

    print(f'  {label_name:<12} horizon={horizon:>4} ticks | {pct_up:.1f}% UP | '
          f'dropping last {horizon} rows (no valid future price)')

print()
print('⚠️  Labels ONLY belong in y arrays — never feed them as input features X.')
print('    Trailing rows with -1 (unknown) are dropped in Cell 11.')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11: Feature Assembly + NaN/Inf Purge
#
# WHY THIS CELL IS CRITICAL:
#
#   Polars `drop_nulls()` removes rows where any value is Polars `null`.
#   It does NOT remove rows where a value is IEEE 754 `NaN` (Not-a-Number).
#   These are DIFFERENT types in Polars:
#     • null  = missing value marker (like pandas NA)
#     • NaN   = floating-point "Not a Number" (e.g., 0.0/0.0, or rolling_std on constant)
#
#   Sources of NaN in this pipeline:
#     1. rolling_zscore(): first ~1000 rows have NaN (window not full yet).
#        fill_null(strategy='forward') fills Polars nulls but NOT float NaN.
#     2. kyle_lambda: OLS on early window with std(X)=0 → NaN slope.
#     3. autocorrelation: np.corrcoef on early constant window → NaN.
#     4. Labels: last 3001 rows have NaN (no future price exists).
#
#   CONSEQUENCE OF NOT FIXING:
#     np.mean() on any array containing a single NaN returns NaN.
#     This is how the original code made ALL data appear corrupted.
#     The Transformer's MSE and BCE losses on NaN targets = NaN loss = NaN gradients.
#     Training silently produces a useless model at exactly 50% accuracy.
#
#   THE FIX (defense-in-depth, 4 layers):
#     Layer 1: fill_nan(0.0)    — replaces float NaN with 0.0
#     Layer 2: fill_null(0.0)   — replaces Polars null with 0.0 (belt-and-suspenders)
#     Layer 3: drop_nulls()     — final safety net for any remaining nulls
#     Layer 4: filter unknown labels — drops rows with -1 marker from Cell 10
#
#   WHY 0.0 (not forward-fill)?
#     0.0 is the mean of all Z-scored features (by construction).
#     Replacing NaN with 0.0 means "no signal" for that feature at that tick.
#     This is the correct behavior for warmup rows.
#     Forward-fill would propagate the FIRST valid value backward = wrong.
# ─────────────────────────────────────────────────────────────────────────────

print('=' * 62)
print('  Cell 11: Feature Assembly + NaN/Inf Purge')
print('=' * 62)

print('\n[1] Building feature DataFrame (FIX: pass numpy arrays directly)...')
t0 = time.time()

# FIX BUG: The original used .tolist() on every column.
# For 5M rows this causes a catastrophic memory explosion:
#   numpy float64 (8 bytes) → Python float (~24 bytes)
#   5M rows × 32 bytes excess = ~160 GB temporary memory → OOM crash.
# FIX: Pass numpy arrays directly to pl.DataFrame — zero conversion overhead.
#       Polars accepts numpy arrays natively with zero memory overhead.

df_features = pl.DataFrame({
    'timestamp':        df['timestamp'],
    'symbol':           df['symbol'],
    'mid_price':        mid_prices,              # numpy array, NOT .tolist()
    'spread':           df['spread'],
    'wofi':             wofi_values,             # numpy array, NOT .tolist()
    'hawkes_intensity': hawkes_intensity,        # numpy array, NOT .tolist()
    'kyle_lambda':      kyle_lambda,             # numpy array, NOT .tolist()
    'amihud_illiq':     amihud_illiq,            # numpy array, NOT .tolist()
    'realized_vol':     realized_vol,            # numpy array, NOT .tolist()
    'autocorrelation':  autocorr_values,         # numpy array, NOT .tolist()
    'wofi_z':           wofi_z,                  # numpy array, NOT .tolist()
    'hawkes_z':         hawkes_z,                # numpy array, NOT .tolist()
    'kyle_lambda_z':    kyle_lambda_z,           # numpy array, NOT .tolist()
    'amihud_z':         amihud_z,                # numpy array, NOT .tolist()
    'spread_z':         spread_z,                # numpy array, NOT .tolist()
    'label_5s':         labels['label_5s'],      # numpy array, NOT .tolist()
    'label_30s':        labels['label_30s'],     # numpy array, NOT .tolist()
    'label_5min':       labels['label_5min'],    # numpy array, NOT .tolist()
})

print(f'  Initial shape: {df_features.shape}')
print(f'  Build time:    {time.time()-t0:.1f}s')

# ── LAYER 1: Replace float NaN with 0.0 ───────────────────────────────────
# This is the primary fix. Polars drop_nulls() MISSES these.
print('\n[2] Layer 1 — fill_nan(0.0): replacing IEEE 754 NaN with 0.0...')
float_cols = [c for c in df_features.columns
              if df_features[c].dtype in (pl.Float32, pl.Float64)]
nan_before = sum(df_features[c].is_nan().sum() for c in float_cols)
print(f'  Float NaN values before: {nan_before:,}')
df_features = df_features.fill_nan(0.0)
nan_after = sum(df_features[c].is_nan().sum() for c in float_cols)
print(f'  Float NaN values after:  {nan_after:,}  ← must be 0')
assert nan_after == 0, f'fill_nan(0.0) failed! Still {nan_after:,} NaN values.'
print('  ✅ fill_nan(0.0) complete — zero float NaN remaining')

# ── LAYER 2: Replace Polars null with 0.0 ──────────────────────────────────
print('\n[3] Layer 2 — fill_null(0.0): replacing Polars null with 0.0...')
null_before = df_features.null_count().sum().item()
print(f'  Polars null values before: {null_before:,}')
df_features = df_features.fill_null(0.0)
null_after = df_features.null_count().sum().item()
print(f'  Polars null values after:  {null_after:,}  ← must be 0')
assert null_after == 0, f'fill_null(0.0) failed! Still {null_after:,} null values.'
print('  ✅ fill_null(0.0) complete — zero Polars null remaining')

# ── LAYER 3: drop_nulls() as final safety net ──────────────────────────────
print('\n[4] Layer 3 — drop_nulls(): final safety net...')
rows_before = len(df_features)
df_features = df_features.drop_nulls()
rows_dropped = rows_before - len(df_features)
print(f'  Rows before drop_nulls(): {rows_before:,}')
print(f'  Rows dropped:             {rows_dropped:,}')
print(f'  Rows after drop_nulls():  {len(df_features):,}')
print('  ✅ drop_nulls() complete')

# ── LAYER 4: Drop rows with unknown labels (-1 marker from Cell 10) ────────
# label_5min has horizon=3000, so last 3000 rows have no valid future price.
# These were marked as -1 in Cell 10. We filter them out to ensure
# the model never trains on rows with unknown labels.
print('\n[5] Layer 4 — Dropping rows with unknown labels (-1 marker)...')
longest_horizon = max(HORIZONS.values())  # 3000

# Method A: Drop last N rows by head (guaranteed to remove all unknown labels)
df_features = df_features.head(len(df_features) - longest_horizon)
rows_after_drop = len(df_features)
print(f'  Dropped last {longest_horizon} rows (no valid 5min label): {rows_after_drop:,} remaining')

# Method B: Also filter by explicit -1 check (belt-and-suspenders)
# This catches any other source of -1 labels
rows_before_filter = len(df_features)
df_features = df_features.filter(
    pl.col('label_5s').cast(pl.Int16) != -1,
    pl.col('label_30s').cast(pl.Int16) != -1,
    pl.col('label_5min').cast(pl.Int16) != -1,
)
rows_after_filter = len(df_features)
extra_dropped = rows_before_filter - rows_after_filter
if extra_dropped > 0:
    print(f'  Additional rows filtered (extra -1 labels): {extra_dropped:,}')
print(f'  Final row count: {rows_after_filter:,}')

# ── VERIFICATION: Confirm zero NaN/null in all numeric columns ─────────────
print('\n[6] Final verification — checking for any remaining NaN/null/Inf...')
float_cols = [c for c in df_features.columns
              if df_features[c].dtype in (pl.Float32, pl.Float64)]

total_nan  = sum(df_features[c].is_nan().sum() for c in float_cols)
total_null = df_features.null_count().sum().item()
total_inf  = sum(
    ((df_features[c].is_infinite()) & (df_features[c].is_not_nan())).sum()
    for c in float_cols
)

print(f'  Float columns checked: {len(float_cols)}')
print(f'  Remaining NaN:         {total_nan:,}   ← must be 0')
print(f'  Remaining null:        {total_null:,}  ← must be 0')
print(f'  Remaining Inf:         {total_inf:,}   ← must be 0')

assert total_nan  == 0, f'CRITICAL: {total_nan:,} NaN values remain after purge!'
assert total_null == 0, f'CRITICAL: {total_null:,} null values remain after purge!'
assert total_inf  == 0, f'CRITICAL: {total_inf:,} Inf values remain after purge!'

# ── Sanity: verify np.mean() no longer returns nan ─────────────────────────
print('\n[7] Sanity check: np.mean() on all feature arrays...')
for col in ['wofi_z', 'hawkes_z', 'kyle_lambda_z', 'amihud_z', 'spread_z']:
    arr  = df_features[col].to_numpy()
    mean = np.mean(arr)
    assert not np.isnan(mean), f'np.mean({col}) = NaN! purge failed.'
    print(f'  np.mean({col:<16}) = {mean:+.6f}  ✅')

print()
print(f'✅ Feature DataFrame assembled in {time.time()-t0:.1f}s')
print(f'   Final shape: {df_features.shape}')
print(f'   All NaN/null/Inf purged. np.mean() safe on all feature columns.')
print()
print('=' * 62)
print('  Cell 11 COMPLETE — data is clean and safe for model training')
print('=' * 62)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Save to parquet and verify round-trip integrity
# ─────────────────────────────────────────────────────────────────────────────

import os
import time
import polars as pl

os.makedirs('/content/drive/MyDrive/AlphaLOB', exist_ok=True)
PARQUET_OUT = '/content/drive/MyDrive/AlphaLOB/lob_features.parquet'

t0 = time.time()

# FIX BUG: use_pyarrow is NOT a valid parameter in pl.DataFrame.write_parquet.
# Valid parameters are: compression, row_group_size, use_arrow_dict, statistics.
# The invalid parameter caused a TypeError at save time.
df_features.write_parquet(PARQUET_OUT, compression='snappy')

file_mb = os.path.getsize(PARQUET_OUT) / 1e6
elapsed = time.time() - t0

# Round-trip verification
df_verify = pl.read_parquet(PARQUET_OUT)
assert len(df_verify) == len(df_features), 'Row count mismatch!'
assert df_verify.columns == df_features.columns, 'Column mismatch!'
del df_verify

print(f'✅ Saved to {PARQUET_OUT}')
print(f'   File size: {file_mb:.0f} MB')
print(f'   Save time: {elapsed:.1f}s')
print(f'   Round-trip verified ✅')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visualize raw vs Z-score-normalized feature distributions
# ─────────────────────────────────────────────────────────────────────────────

import matplotlib
matplotlib.use('Agg')  # non-interactive backend
import matplotlib.pyplot as plt

print('Plotting feature distributions...')
sample = df_features.sample(min(10_000, len(df_features)), seed=42)

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('AlphaLOB — Feature Distributions: Raw vs Z-Score Normalized',
             fontsize=13, fontweight='bold')

raw_cols  = ['wofi',   'hawkes_intensity', 'kyle_lambda',    'amihud_illiq']
norm_cols = ['wofi_z', 'hawkes_z',         'kyle_lambda_z',  'amihud_z']
titles    = ['WOFI',   'Hawkes λ(t)',       "Kyle's Lambda",  'Amihud ILLIQ']
colors    = ['#2196F3','#4CAF50',           '#FF9800',        '#9C27B0']

for j, (raw, norm, title, color) in enumerate(zip(raw_cols, norm_cols, titles, colors)):
    # Raw distribution (top row)
    axes[0, j].hist(sample[raw].to_numpy(), bins=60, color=color, alpha=0.8)
    axes[0, j].set_title(f'{title} (raw)')
    axes[0, j].grid(alpha=0.3)

    # Z-score distribution (bottom row)
    axes[1, j].hist(sample[norm].to_numpy(), bins=60, color=color, alpha=0.8)
    axes[1, j].set_title(f'{title} (z-score)')
    axes[1, j].axvline(0,  color='red',  linestyle='--', alpha=0.6, label='μ=0')
    axes[1, j].axvline(-3, color='gray', linestyle=':',  alpha=0.5)
    axes[1, j].axvline(+3, color='gray', linestyle=':',  alpha=0.5, label='±3σ')
    axes[1, j].set_xlim(-5, 5)
    axes[1, j].legend(fontsize=7)
    axes[1, j].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/features_overview.png', dpi=150, bbox_inches='tight')
plt.close()
print('✅ Feature distributions saved → /content/features_overview.png')

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Label balance check — verify approximately 50/50 split
# ─────────────────────────────────────────────────────────────────────────────

print('=== LABEL BALANCE CHECK ===')
for lbl in ['label_5s', 'label_30s', 'label_5min']:
    # Only count valid labels (1=UP, 0=DOWN), exclude -1 (unknown)
    col_data = df_features[lbl].cast(pl.Int16).to_numpy()
    valid_mask = col_data >= 0           # exclude -1 unknown labels
    up   = int((col_data[valid_mask] == 1).sum())
    down = int((col_data[valid_mask] == 0).sum())
    total = valid_mask.sum()
    pct   = up / total * 100 if total > 0 else 0
    ok    = '✅' if 45 <= pct <= 55 else '⚠️'
    print(f'  {ok} {lbl:<12}: UP={up:,} ({pct:.1f}%) | '
          f'DOWN={down:,} ({100-pct:.1f}%) | valid rows={total:,}')

print()
print('  Labels should be ~50/50. If badly skewed, add class_weight to the loss in Notebook 03.')

In [ ]:
print()
print('=' * 58)
print('  NOTEBOOK 02 COMPLETE — FEATURE ENGINEERING')
print('=' * 58)
print(f'  Output: {PARQUET_OUT}')
print(f'  Rows:   {len(df_features):,}')
print(f'  Shape:  {df_features.shape}')
print()
print('  Features computed:')
print('    ✅ WOFI           — deque O(1) rolling window')
print('    ✅ Hawkes λ(t)    — MLE fit once, recursive apply')
print("    ✅ Kyle's Lambda  — rolling OLS 5-min (statsmodels)")
print('    ✅ Amihud ILLIQ   — rolling |ret|/dollar_vol')
print('    ✅ Z-Score norm   — rolling 1000-tick window, no look-ahead')
print('    ✅ Labels         — 3 horizons (5s, 30s, 5min)')
print('    ✅ NaN/Inf purge  — 4-layer defense')
print()
print('  Bugs fixed in this notebook:')
print('    🔴 np.diff(prepend=x) bug → manual first-diff')
print('    🔴 .tolist() on 5M rows → OOM → pass numpy arrays directly')
print('    🔴 use_pyarrow=True → invalid param → removed')
print('    🟡 np.corrcoef unguard → NaN propagation → added guards')
print('    🟡 Unknown labels → fake 0 → marked -1, filtered out')
print()
print('  Next step → Run 03_train_lobtransformer.ipynb')
print('=' * 58)